# PAX Fabric Notebook (PAX -> Lakehouse Tables)

Pulls Purview Audit + Entra data directly from Microsoft Graph and writes
**Delta tables** under `Tables/<TargetSchema>/` in the default lakehouse.
**No persistent CSV output** — the CSV bundle from Phase A0 / Phase A is
not produced. (A transient scratch directory is used internally so the
existing rollup / Entra processors can be reused; it is deleted at end
of run unless `KeepScratch=True`.)

## Prerequisites
- Default lakehouse attached (schema-enabled), with the target schema
  (default `dbo`) already created.
- **`pax_fabric/` package is pulled from GitHub at run time** — no manual
  upload to `Files/code/` is required. The clone cell below does a fresh
  shallow clone of `GitHubRepoUrl` at `GitHubRef` into a temp dir on
  every run, so each execution is free of stale `.pyc` from prior runs.
  The temp dir is removed by the cleanup cell at the end.
- **`GitHubRef` accepts either of two values** — the clone cell
  auto-detects which one you supplied:
  - **Branch name** — fetches the latest commit on the branch.
  - **Commit SHA** — fetches that exact
    commit. SHA refs use a `clone --no-checkout` + `fetch --depth 1` +
    `checkout --detach` sequence because `git clone --branch` does not
    accept commit IDs.
  For production, pin a commit SHA so re-runs are byte-identical.
- A **variable library** is attached to the workspace and supplies the
  parameters consumed by the parameters cell below (auth identifiers,
  Key Vault location, date range, run-control flags, and the GitHub
  repo URL + ref). The client secret is fetched from Key Vault via
  `notebookutils.credentials.getSecret(KeyVaultUri, KeyVaultSecret)`.
- The repo + ref in `GitHubRepoUrl` / `GitHubRef` must be reachable from
  the Fabric runtime (public repo, or private repo with credentials baked
  into the URL / the runtime's git config). For commit-SHA refs, the
  remote must allow fetching arbitrary SHAs — GitHub does by default.
- `git` is on `PATH` in the Fabric Python runtime (it is, by default).
- `deltalake>=0.17` available (auto-installed by the clone cell if
  missing).

## BYOD (Bring-Your-Own-Data)
When you already have a Purview audit export and want to skip the live
Graph collection, supply the pre-collected CSV via the `PurviewInputFile`
parameter (or `PurviewInputTable` for a lakehouse Delta table). BYOD
replaces the Graph API call — everything downstream (dedup, structure,
rollup, dashboard shaping, retention, deidentify) runs unchanged.

- **Path format** — lakehouse-relative works directly, e.g.
  `Files/byod/audit.csv` is auto-resolved to
  `/lakehouse/default/Files/byod/audit.csv`. Absolute paths and
  `abfss://` URIs are also accepted.
- **Required CSV columns** — `RecordId`, `CreationDate`, `RecordType`,
  `Operation`, `UserId`, `AuditData` (case-sensitive header;
  `AuditData` must be a JSON object per row).
- **Activation** — BYOD only kicks in when the run targets something
  that consumes audit data: pass `Rollup=True`, `RollupPlusRaw=True`, or
  a `Dashboard` value (`AIO`, `M365`, or `ValueLens`). If none of those
  are requested, the supplied input is ignored and a NOTE is logged.
- **StartDate / EndDate** — ignored in BYOD mode. The full supplied
  source is processed; date-trim only applies to the live path.
- **AppRegistration credential** — not required when the run is fully
  offline (BYOD without `IncludeUserInfo=True`); the auth gate skips
  the Key Vault fetch. If `IncludeUserInfo=True` is also set, Entra
  still needs the Graph credential.
- **Read-only** — the supplied file is streamed once into per-run
  JSONL shards; the caller's file is never modified.
- **Checkpoint** — disabled for BYOD (a single-shot ingest has nothing
  to resume from).

## Output
- Delta tables written under `Tables/<TargetSchema>/` in the default
  lakehouse. Which tables are produced depends on the run flags:
  - **Audit raw** — always written when audit ingest runs.
  - **CopilotInteraction rollup family** — `..._Rollup`,
    plus `..._UserStats`, `..._SessionCohort`, and `..._SessionStats`
    when `IncludeM365Usage=True`.
  - **Entra users** — `Entra_Users` when `IncludeUserInfo=True`.
- **Append semantics:** every run appends new rows to the same target
  table per logical dataset. The function returns one entry per
  written table in `result["delta_tables"]` with `rows_written` and
  `is_init` (true on the first write that creates the table).
- **Schema evolution (mod16):**
  - **Additive drift** — new columns in the incoming batch are
    absorbed automatically via `schema_mode='merge'`; existing rows
    get NULL for the new column.
  - **Destructive drift** — when columns the Delta table already has
    are missing from the incoming batch, the auto-drop path drops
    those columns from the Delta table (regardless of how many) and
    then appends. Delta time-travel preserves the pre-drop version
    for ~7 days, so an accidental drop is recoverable via
    `DeltaTable(uri).restore(version=N)`.
- **Optional retention** (`RetentionDays`) prunes rows older than the
  cutoff from each Delta table that has a `CreationDate` column. When
  retention deletes any rows from a `_Rollup` table and
  `IncludeM365Usage=True`, `UserStats` and `SessionCohort` are
  recomputed from the trimmed Rollup so percentiles, tiers, and
  active-day counts reflect only the retained window.

## Gate
The verify cell reads each written Delta table back and asserts the
post-write row count is **at least** `rows_written` (i.e. the table
exists, is readable, and contains the rows we just appended).

In [ ]:
# ---- Parameters (layered: pipeline > Variable Library > pax_run defaults) ---
#
# HOW PIPELINE PARAMETERS FLOW IN FABRIC
# --------------------------------------
# Fabric uses papermill-style injection: when the Notebook activity runs,
# Fabric auto-generates a hidden cell that assigns each `Base parameter`
# as a top-level Python variable (e.g. `StartDate = "2026-06-01"`)
# inserted at the VERY TOP of the notebook when no cell is tagged
# `parameters`.
#
# IMPORTANT: DO NOT toggle the `parameters` tag on this cell. If tagged,
# Fabric injects AFTER this cell, which would clobber overrides.
#
# WHERE VALUES COME FROM
# ----------------------
# Variable Library (env constants):     infra + TargetSchema + KeepScratch
#                                       + MaxConcurrency + PartitionHours
#                                       + RetentionDays.
# Pipeline Base Parameters (run shape): StartDate, EndDate, NumberOfDays,
#                                       Rollup, IncludeM365Usage,
#                                       IncludeUserInfo, OnlyUserInfo,
#                                       IncludeAgent365Info, OnlyAgent365Info,
#                                       GroupNames, plus any ad-hoc knob.
#
# WHAT GETS FORWARDED TO pax_run()
# --------------------------------
# * Mandatory infra (auth, Key Vault, GitHub source) — always sent
#   explicitly; NEVER in `pipeline_overrides`.
# * EVERY other pipeline-injected variable — forwarded verbatim. Add a
#   new `Base parameter` on the pipeline (e.g. `AgentId`, `ActivityTypes`)
#   and it flows through with no code change. Unknown keys are silently
#   ignored by pax_run.config_from_params(), so mistyped names are safe
#   too.
# * VarLib env constants — merged in as fallbacks so pax_run sees them
#   even when the pipeline didn't send them. Pipeline wins over VarLib.
# * Empty strings from either source are treated as 'not sent' so blank
#   defaults (e.g. StartDate="" or GroupNames="") don't override pax_run's
#   own defaults. GroupNames accepts a comma-separated string.
# * NumberOfDays is consumed HERE (not forwarded): when >0 AND
#   StartDate/EndDate are absent, we derive a rolling window ending
#   today (UTC). Explicit StartDate/EndDate always win — set either one
#   on the pipeline run and NumberOfDays is ignored.

# ---- Auto-capture every pipeline-injected variable ---------------------
# globals() at this point contains only Fabric-injected params + kernel
# built-ins (which we filter out).
_INFRA_NAMES = {
    "TenantId", "ClientId", "KeyVaultUri", "KeyVaultSecret",
    "GitHubRepoUrl", "GitHubRef", "ClientSecret",
}
_KERNEL_GLOBALS = {"In", "Out", "exit", "quit", "get_ipython", "open"}

def _is_forwardable(value):
    if value is None or not isinstance(value, (str, int, float, bool, list, dict)):
        return False
    if isinstance(value, str) and value == "":
        return False
    return True

_pipeline_captured = {}
for _name, _value in list(globals().items()):
    if _name.startswith("_") or _name in _KERNEL_GLOBALS or _name in _INFRA_NAMES:
        continue
    if not _is_forwardable(_value):
        continue
    _pipeline_captured[_name] = _value

# ---- Derive StartDate/EndDate from NumberOfDays (rolling window) -------
# NumberOfDays lets scheduled runs cover a trailing N-day window without
# editing dates on every trigger (e.g. NumberOfDays=1 -> daily refresh).
# Explicit StartDate or EndDate from the pipeline always take priority.
_number_of_days = _pipeline_captured.pop("NumberOfDays", None)
_user_set_dates = (
    "StartDate" in _pipeline_captured or "EndDate" in _pipeline_captured
)
if (
    isinstance(_number_of_days, int)
    and not isinstance(_number_of_days, bool)
    and _number_of_days > 0
    and not _user_set_dates
):
    from datetime import datetime, timedelta, timezone
    _end = datetime.now(timezone.utc).date()
    _start = _end - timedelta(days=_number_of_days)
    _pipeline_captured["StartDate"] = _start.isoformat()
    _pipeline_captured["EndDate"] = _end.isoformat()
    print(
        f"NumberOfDays={_number_of_days} -> "
        f"StartDate={_pipeline_captured['StartDate']}, "
        f"EndDate={_pipeline_captured['EndDate']}"
    )
elif _number_of_days and _user_set_dates:
    print(
        f"NumberOfDays={_number_of_days} ignored — explicit StartDate/EndDate "
        "supplied by pipeline take priority."
    )

# ---- Resolve mandatory infra (pipeline > Variable Library) -------------
import notebookutils

_vl = notebookutils.variableLibrary.getLibrary("FTGI_VarLib_Dev")

def _pipeline_sent(name):
    """Return the pipeline-injected value for `name`, or None if unset/blank."""
    if name not in globals():
        return None
    v = globals()[name]
    if isinstance(v, str) and v == "":
        return None
    return v

TenantId       = _pipeline_sent("TenantId")       or _vl.TenantId
ClientId       = _pipeline_sent("ClientId")       or _vl.ClientId
KeyVaultUri    = _pipeline_sent("KeyVaultUri")    or _vl.KeyVaultUri
KeyVaultSecret = _vl.KeyVaultSecret
GitHubRepoUrl  = _pipeline_sent("GitHubRepoUrl")  or _vl.GitHubRepoUrl
GitHubRef      = _pipeline_sent("GitHubRef")      or _vl.GitHubRef

# ---- Layer VarLib env constants under pipeline overrides ---------------
# Add a name here if a new env-constant variable is added to VarLib.
_VL_ENV_KEYS = (
    "TargetSchema", "KeepScratch", "MaxConcurrency",
    "PartitionHours", "RetentionDays",
)

pipeline_overrides = {}
for _key in _VL_ENV_KEYS:
    try:
        _val = getattr(_vl, _key)
    except AttributeError:
        continue
    if _is_forwardable(_val):
        pipeline_overrides[_key] = _val
pipeline_overrides.update(_pipeline_captured)  # pipeline wins over VarLib

# ---- Values referenced by later cells (retention / recompute) ----------
TargetSchema     = pipeline_overrides.get("TargetSchema", "dbo")
IncludeM365Usage = pipeline_overrides.get("IncludeM365Usage", False)
RetentionDays    = pipeline_overrides.get("RetentionDays")

# ---- Resolve client secret from Key Vault ------------------------------
ClientSecret = notebookutils.credentials.getSecret(KeyVaultUri, KeyVaultSecret)
if not ClientSecret:
    raise RuntimeError(
        f"Failed to fetch secret '{KeyVaultSecret}' from {KeyVaultUri}. "
        "Verify Key Vault access and secret name."
    )


In [ ]:
# ---- Clone pax_fabric from GitHub (public repo) ----
import os, re, sys, subprocess, tempfile, importlib

# Fresh clone each run — no stale .pyc, deterministic per GitHubRef
CodePath = tempfile.mkdtemp(prefix="pax_fabric_repo_")

# git clone --branch only accepts branch/tag names. For a commit SHA we
# clone with no checkout, then fetch the SHA shallowly and check it out.
_is_commit_sha = bool(re.fullmatch(r"[0-9a-fA-F]{7,40}", GitHubRef or ""))

if _is_commit_sha:
    subprocess.check_call(
        ["git", "clone", "--filter=blob:none", "--no-checkout", GitHubRepoUrl, CodePath],
        stdout=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ["git", "-C", CodePath, "fetch", "--depth", "1", "origin", GitHubRef],
        stdout=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ["git", "-C", CodePath, "checkout", "--detach", GitHubRef],
        stdout=subprocess.DEVNULL,
    )
else:
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", GitHubRef, GitHubRepoUrl, CodePath],
        stdout=subprocess.DEVNULL,
    )

print(f"Cloned {GitHubRepoUrl}@{GitHubRef} -> {CodePath}")

# pax_fabric/ is at the repo root, so the parent dir on sys.path is CodePath itself
PackageParent = CodePath

# Evict any prior pax_fabric import so a re-run picks up the new clone
for _m in [m for m in list(sys.modules) if m == "pax_fabric" or m.startswith("pax_fabric.")]:
    del sys.modules[_m]

# ---- Dependencies: ensure deltalake is available ----
try:
    import deltalake  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "deltalake>=0.17"]
    )
    importlib.invalidate_caches()
    import deltalake  # noqa: F401

import pyarrow  # noqa: F401
import pandas   # noqa: F401
print("deltalake version:", deltalake.__version__)



In [ ]:
# ---- Invoke pax_fabric.run() ----
import sys

if CodePath and CodePath not in sys.path:
    sys.path.insert(0, CodePath)

from pax_fabric import run as pax_run, files_io  # noqa: E402

# Auth + output mode are always required. User-tunable knobs are spread
# in only when the pipeline actually sent them, so unset flags don't
# collide with the OnlyUserInfo / IncludeM365Usage validators inside
# pax_run.config_from_params().
params = {
    "Auth":         "AppRegistration",
    "TenantId":     TenantId,
    "ClientId":     ClientId,
    "ClientSecret": ClientSecret,
    "OutputMode":   "delta",
    **pipeline_overrides,
}

result = pax_run(params)

print(f"success       : {result['success']}")
print(f"run_id        : {result['run_id']}")
print(f"target_schema : {result['target_schema']}")
print(f"records       : {result['records_fetched']}")
print(f"output_rows   : {result['output_rows']}")
print(f"elapsed (s)   : {result['elapsed_seconds']}")
print(f"log_file      : {result['log_file']}")

# ---- v1.11.16 feature surface ------------------------------------------
# Only shown when the corresponding knob was engaged, so legacy default
# runs (Watermark=False, no BYOD, UserHistory=Off, EmitMetricsJson=False)
# keep byte-identical stdout. Fields are stamped by pipeline.run() when
# active; absent otherwise.
_v1116_active = bool(
    result.get("watermark_enabled")
    or result.get("byod_source")
    or (result.get("user_history") and result.get("user_history") != "Off")
    or result.get("metrics_json_path")
)
if _v1116_active:
    print(f"script_ver    : {result.get('script_version') or '(unknown)'}")
    if result.get("watermark_enabled"):
        _wcs = result.get("watermark_window_start") or "(bootstrap)"
        _wce = result.get("watermark_window_end") or "(unknown)"
        print(f"watermark     : enabled   (window {_wcs} -> {_wce})")
    if result.get("byod_source"):
        print(
            f"byod          : {result['byod_source']}  "
            f"({result.get('byod_records_loaded', 0):,} records)"
        )
    _uh = result.get("user_history")
    if _uh and _uh != "Off":
        _hed = result.get("history_effective_date") or "(auto)"
        _src = result.get("history_effective_date_source") or "derived"
        print(f"user_history  : {_uh}  (HED={_hed}, src={_src})")
    if result.get("metrics_json_path"):
        print(f"metrics_json  : {result['metrics_json_path']}")

print()
print("Forwarded knobs:", sorted(pipeline_overrides.keys()) or "(none — pax_run defaults apply)")
print()
print("Delta tables written:")
for t in result.get("delta_tables", []):
    print(f"  - {t['table']:<50} rows={t['rows_written']:<8} path={t['path']}")

if not result["success"]:
    raise RuntimeError(f"pax_run failed: {result.get('error')}")

In [ ]:
# ---- Phase B gate: read each Delta table back and confirm rows are present ----
from deltalake import DeltaTable

storage_options = files_io.onelake_storage_options()

failures = []
print(f"{'TABLE':<55} {'WRITTEN':>10} {'DELTA_TOTAL':>12}  STATUS")
print("-" * 92)
for t in result.get("delta_tables", []):
    written = t["rows_written"]
    try:
        dt = DeltaTable(t["path"], storage_options=storage_options)
        total_rows = dt.to_pyarrow_dataset().count_rows()
    except Exception as exc:
        print(f"{t['table']:<55} {written:>10}   ERROR reading back: {exc}")
        failures.append(t["table"])
        continue
    # Post-append total must be >= what we just wrote. On first ingest
    # (t['is_init'] == True) this is equality; on subsequent runs it is
    # strictly greater because prior rows accumulate.
    flag = "OK" if total_rows >= written else "MISMATCH"
    init_tag = " [append-init]" if t.get("is_init") else ""
    print(f"{t['table']:<55} {written:>10} {total_rows:>12}  {flag}{init_tag}")
    if total_rows < written:
        failures.append(t["table"])

if failures:
    raise AssertionError(
        f"Phase B gate FAILED for {len(failures)} table(s): {failures}"
    )

print()
print("Phase B gate: all tables readable with row count >= rows written this run.")

In [ ]:
# ---- Data Retention: prune rows older than RetentionDays ----
if RetentionDays is not None and RetentionDays > 0:
    from pax_fabric.mod17_pax_retention import enforce_retention
 
    storage_options = files_io.onelake_storage_options()
 
    print(f"Data retention: deleting rows with CreationDate older than {RetentionDays} days")
    print()
    print(f"{'TABLE':<55} {'BEFORE':>10} {'AFTER':>10} {'DELETED':>10}  STATUS")
    print("-" * 100)
 
    retention_failures = []
    _rollup_rows_deleted = 0
    _hed = result.get("history_effective_date") or None
    for t in result.get("delta_tables", []):
        ret = enforce_retention(
            target_uri=t["path"],
            retention_days=RetentionDays,
            storage_options=storage_options,
            history_effective_date=_hed,
        )
 
        if ret["skipped"]:
            print(f"{t['table']:<55} {'—':>10} {'—':>10} {'—':>10}  SKIPPED (no CreationDate)")
            continue
 
        if not ret["success"]:
            print(f"{t['table']:<55} {ret['rows_before']:>10} {'—':>10} {'—':>10}  ERROR: {ret['error']}")
            retention_failures.append(t["table"])
            continue
 
        print(
            f"{t['table']:<55} {ret['rows_before']:>10} {ret['rows_after']:>10} "
            f"{ret['rows_deleted']:>10}  OK (cutoff: {ret['cutoff_date']})"
        )
 
        # Track if Rollup Delta had rows deleted (triggers recompute below)
        if "_Rollup" in t["table"] and "UserStats" not in t["table"] and "SessionCohort" not in t["table"] and "SessionStats" not in t["table"]:
            _rollup_rows_deleted += ret.get("rows_deleted", 0)
 
    if retention_failures:
        print()
        print(f"⚠ Retention failed for {len(retention_failures)} table(s): {retention_failures}")
    else:
        print()
        print(f"Data retention complete (cutoff: {RetentionDays} days).")
 
    # ---- Post-retention recompute: refresh UserStats & SessionCohort ----
    # After retention trims old rows from Rollup/SessionStats Delta, the
    # UserStats and SessionCohort tables are stale (computed from pre-trim
    # data). Recompute them from the trimmed Rollup so percentiles, tiers,
    # and active-day counts reflect only the retained window.
    # Only runs when: Rollup rows were actually deleted AND M365 usage mode.
    if _rollup_rows_deleted > 0 and IncludeM365Usage:
        print()
        print(f"Post-retention recompute: Rollup had {_rollup_rows_deleted:,} rows deleted — "
              "refreshing UserStats & SessionCohort from trimmed data...")
        from pax_fabric.pipeline import _recompute_userstats_from_delta
        recomputed = _recompute_userstats_from_delta(
            delta_results=result.get("delta_tables", []),
            schema=TargetSchema,
            log_fn=lambda msg, lvl="INFO": print(f"  {msg}"),
        )
        if recomputed:
            print(f"Post-retention recompute: {len(recomputed)} table(s) refreshed.")
            for r in recomputed:
                print(f"  {r['table']:<55} {r['rows_written']:>10} rows  [recomputed]")
        else:
            print("Post-retention recompute: no tables refreshed (check logs above for warnings).")
else:
    print("Data retention: disabled (RetentionDays is None or 0).")

In [ ]:
# ---- Cleanup: remove the GitHub clone temp dir ----
import os, shutil

if 'CodePath' in dir() and CodePath and os.path.isdir(CodePath):
    try:
        shutil.rmtree(CodePath)
        print(f"Cleanup: removed temp clone dir {CodePath}")
    except Exception as exc:
        print(f"Cleanup: failed to remove {CodePath} -> {exc}")
else:
    print("Cleanup: no temp clone dir to remove (already gone or not set).")